# Urban Runoff: Workflow for Resample, Masking and Watershed Calculation

Maddie Berger  
12-09-2024
Updated: 09-20-2025

This script contains the python code used to run on the KOA super computer for big jobs. Once it works here, we can export the cell chunks and .py scripts and upload to KOA. 
Conda environment I use for this script is `geoenv`.

Inputs: 
- Tiled raw impervious surface rasters for 2011, 2017 and 2021
- Analysis mask at 200 m and 500 m
- Watersheds for zonal statistics

Outputs:
- Masked tiles
- Table per raster of zonal statistics (ie area of impervious per watershed)




## Set up 

In [ ]:
# Set up Packages and Dropbox file paths

import os
import rasterio
import rioxarray
import geopandas as gpd
import pandas as pd
import xarray as xr
import dask
from shapely.geometry import mapping


# Define Dropbox paths
print(os.path.exists("C:\\Users\\mmtb\\Donovan Lab Dropbox"))

urban_dropbox = dropbox_base = r"C:\Users\mmtb\Donovan Lab Dropbox\Donovan Lab Team Folder\Donovan_Lab_GIS\Drivers\2_Urban_Runoff\large-raw-data"
raster_folder = os.path.join(urban_dropbox,"ccap2011","tiled")
rasters_out = os.path.join(urban_dropbox,"ccap2011","masked")

raster_folder_2017 = os.path.join(urban_dropbox,"ccap2017","tiled")
raster_folder_2021 = os.path.join(urban_dropbox,"ccap2021","tiled")

# Define local paths
base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
mask_vector_path = os.path.join(base_dir,"data","analysis_mask_500_wNiihau.shp")
sheds_vector_path = os.path.join(base_dir, "data", "WBD_HU12_n83utm4_2023fixed.shp")
#watersheds_vector_path = "/path/to/watersheds.shp"

True


In [ ]:
# Test raster folder (Dropbox - input)
print("✅ Testing raster_folder:")
print("  Path:", raster_folder)
print("  Exists:", os.path.exists(raster_folder))
if os.path.exists(raster_folder):
    raster_files = [f for f in os.listdir(raster_folder) if f.endswith(".TIF")]
    print(f"  Found {len(raster_files)} .tif files")
    print("  First 5 files:", raster_files[:5])

# Test rasters_out folder (Dropbox - output)
print("\n✅ Testing rasters_out:")
print("  Path:", rasters_out)
print("  Exists:", os.path.exists(rasters_out))


✅ Testing raster_folder:
  Path: C:\Users\mmtb\Donovan Lab Dropbox\Donovan Lab Team Folder\Donovan_Lab_GIS\Drivers\2_Urban_Runoff\large-raw-data\ccap2011\tiled
  Exists: True
  Found 8 .tif files
  First 5 files: ['ccap2011_rast10.TIF', 'ccap2011_rast11.TIF', 'ccap2011_rast12.TIF', 'ccap2011_rast13.TIF', 'ccap2011_rast3.TIF']

✅ Testing rasters_out:
  Path: C:\Users\mmtb\Donovan Lab Dropbox\Donovan Lab Team Folder\Donovan_Lab_GIS\Drivers\2_Urban_Runoff\large-raw-data\ccap2011\masked
  Exists: True


Test and Load the Vectors (Mask and Watersheds)

In [ ]:
print(mask_vector_path)
if not os.path.exists(mask_vector_path):
    print(f"File not found: {mask_vector_path}")

print(sheds_vector_path)
if not os.path.exists(sheds_vector_path):
    print(f"File not found: {sheds_vector_path}")


c:\Users\mmtb\projects\urban_runoff_local\data\analysis_mask_500_wNiihau.shp
c:\Users\mmtb\projects\urban_runoff_local\data\WBD_HU12_n83utm4_2023fixed.shp


In [ ]:

# Load the vector mask
mask_gdf = gpd.read_file(mask_vector_path)
mask_geometry = mask_gdf.geometry

#Load the watersheds input
sheds_gdf = gpd.read_file(sheds_vector_path)
sheds_geometry = sheds_gdf.geometry
print(sheds_gdf.columns) # get the column with unique IDS for step 3
print(sheds_gdf["HUC12"].is_unique)


Index(['TNMID', 'MetaSource', 'SourceData', 'SourceOrig', 'SourceFeat',
       'LoadDate', 'GNIS_ID', 'AreaAcres', 'AreaSqKm', 'States', 'HUC12',
       'Name', 'HUType', 'HUMod', 'ToHUC', 'NonContrib', 'NonContr_1',
       'Shape_Leng', 'Shape_Area', 'Area_Ag_sq', 'Area_Ag__1', 'geometry'],
      dtype='object')
True


### Check that CRS of the inputs match 

In [ ]:

# Check CRS of the inputs
# Path to the raster file - change this if you change the folder of rasters
raster_samp_2011 = os.path.join(raster_folder,"ccap2011_rast7.TIF")
raster_samp_2017 = os.path.join(raster_folder_2017,"utm4_hi_south_20171.tif")

with rasterio.open(raster_samp_2011) as src:
    raster_crs = src.crs
    resolution = src.res
    print(f"2011 Raster CRS: {raster_crs}")
    print(f"2011 Raster Resolution: {resolution}")
    pixel_width = src.transform.a
    pixel_height = abs(src.transform.e)  # take absolute value
    print("Pixel size:", pixel_width, pixel_height)

with rasterio.open(raster_samp_2017) as src:
    raster_crs = src.crs
    resolution = src.res
    print(f"2017 Raster CRS: {raster_crs}")
    print(f"2017 Raster Resolution: {resolution}")


# vectors

# Load the vector files
#mask = gpd.read_file(mask_vector_path)

# Print the CRS of the vector files
mask_crs = mask_gdf.crs
sheds_crs = sheds_gdf.crs
print(f"Mask CRS: {mask_crs}")
print(f"Sheds CRS: {sheds_crs}")

# check match between mask and the raster file

if raster_crs == mask_crs:
    print("The CRS of the raster and mask files match.")
else:
    print("The CRS of the raster and vector files do not match.")
    print(f"Raster CRS: {raster_crs}")
    print(f"Mask CRS: {mask_crs}")


2011 Raster CRS: EPSG:26904
2011 Raster Resolution: (2.3999999999970005, 2.3999999999970014)
Pixel size: 2.3999999999970005 2.3999999999970014
2017 Raster CRS: EPSG:26904
2017 Raster Resolution: (1.0, 1.0)
Mask CRS: EPSG:26904
Sheds CRS: EPSG:26904
The CRS of the raster and mask files match.


## Step 0 (for 2017 and 2021 only) - Resample to 3 m

In order to make all of these comparable, they will have to be in the same resolution. We can't make the 2011 3m rasters finer, so we'll have to resample the newer rasters, which are at 1m resolution.
I tried to avoid this step but unfortunately the difference in resolution is masking the effect of time - ie we find that all watersheds have MORE impervious surface in 2011 than later years, which doesn't make sense logically and obscures places that changed due to actual development.

https://geowombat.readthedocs.io/en/latest/index.html <- for more on geowombat!

In [ ]:
# try geowombat - note that this is not supported in python 3.13, which is what we are using here

import geowombat as gw

# create file path to folder to save these

resampled_out = os.path.join(urban_dropbox,"ccap2017","resampled")

# get list of rasters we want to resample

raster_list = [f for f in os.listdir(raster_folder) if f.endswith('.TIF')]

# template raster to ensure its the same as 2011

for raster in raster_list:
    raster_path = os.path.join(raster_folder, raster)
    output_path = os.path.join(resampled_out, f"{raster}_3x3")

    if os.path.exists(output_path):
        print(f"Skipping {raster}, already processed.")
        continue

    try:
        # 
        with gw.config.update(ref_res=(3,3)): # this is unclear to me, what units?
            with gw.open(image, resamplong="bilinear") as src:
                print(src)
                src.gw.to_raster(output_path)
        
    except Exception as e:
        print(f"Failed to process {raster}: {e}")



In [ ]:
# try rasterio

import rasterio
from rasterio.enums import Resampling
from rasterio import Affine

target_resolution = pixel_width  # from 2011
source_resolution = 1.0     # assumed resolution of the original raster

scale_factor = source_resolution / target_resolution

with rasterio.open(raster_samp_2017) as dataset:
    data = dataset.read(
        out_shape=(
            dataset.count,
            int(dataset.height * scale_factor),
            int(dataset.width * scale_factor)
            ),
        resampling=Resampling.bilinear
        )

    dst_transform = dataset.transform * Affine.scale(
        (dataset.width / data.shape[-1]),
        (dataset.height / data.shape[-2])
        )
print("Original shape:", dataset.shape)
print("Resampled shape:", data.shape[1:])
print("New pixel size (x, y):", dst_transform.a, abs(dst_transform.e))

Original shape: (53197, 46762)
Resampled shape: (22165, 19484)
New pixel size (x, y): 2.4000205296653663 2.4000451161741485


Loop to apply to all rasters in the folder

In [ ]:
import os
import rasterio
from rasterio.enums import Resampling
from rasterio import Affine

# new file path for resampled rasters
resampled_out = os.path.join(urban_dropbox,"ccap2021","resampled")

# Set this based on your reference raster (already done earlier)
target_resolution = 2.3999
source_resolution = 1.0
scale_factor = source_resolution / target_resolution

raster_list = [f for f in os.listdir(raster_folder_2021) if f.endswith('.tif')]

for raster in raster_list:
    raster_path = os.path.join(raster_folder_2021, raster)
    output_path = os.path.join(resampled_out, f"{raster}_3x3.tif")

    if os.path.exists(output_path):
        print(f"Skipping {raster}, already processed.")
        continue

    try:
        with rasterio.open(raster_path) as dataset:
            data = dataset.read(
                out_shape=(
                    dataset.count,
                    int(dataset.height * scale_factor),
                    int(dataset.width * scale_factor)
                ),
                resampling=Resampling.bilinear
            )

            dst_transform = dataset.transform * Affine.scale(
                dataset.width / data.shape[-1],
                dataset.height / data.shape[-2]
            )

            # Write resampled data to a new file
            with rasterio.open(
                output_path,
                "w",
                driver="GTiff",
                height=data.shape[1],
                width=data.shape[2],
                count=dataset.count,
                dtype=data.dtype,
                crs=dataset.crs,
                transform=dst_transform,
            ) as dest:
                dest.write(data)

            print(f"Processed and saved: {output_path}")

    except Exception as e:
        print(f"Failed to process {raster}: {e}")


Processed and saved: C:\Users\mmtb\Donovan Lab Dropbox\Donovan Lab Team Folder\Donovan_Lab_GIS\Drivers\2_Urban_Runoff\large-raw-data\ccap2021\resampled\utm4_hi_hawaii_2021_ccap_0.tif_3x3.tif
Processed and saved: C:\Users\mmtb\Donovan Lab Dropbox\Donovan Lab Team Folder\Donovan_Lab_GIS\Drivers\2_Urban_Runoff\large-raw-data\ccap2021\resampled\utm4_hi_hawaii_2021_ccap_1.tif_3x3.tif
Processed and saved: C:\Users\mmtb\Donovan Lab Dropbox\Donovan Lab Team Folder\Donovan_Lab_GIS\Drivers\2_Urban_Runoff\large-raw-data\ccap2021\resampled\utm4_hi_hawaii_2021_ccap_2.tif_3x3.tif
Processed and saved: C:\Users\mmtb\Donovan Lab Dropbox\Donovan Lab Team Folder\Donovan_Lab_GIS\Drivers\2_Urban_Runoff\large-raw-data\ccap2021\resampled\utm4_hi_hawaii_2021_ccap_3.tif_3x3.tif
Processed and saved: C:\Users\mmtb\Donovan Lab Dropbox\Donovan Lab Team Folder\Donovan_Lab_GIS\Drivers\2_Urban_Runoff\large-raw-data\ccap2021\resampled\utm4_hi_hawaii_2021_ccap_4.tif_3x3.tif
Processed and saved: C:\Users\mmtb\Donovan La

## Step 1 Simplify Mask Geometry

In [ ]:
# Load and simplify
gdf = gpd.read_file(mask_vector_path)
gdf_simplified = gdf.copy()
gdf_simplified['geometry'] = gdf_simplified.geometry.simplify(tolerance=0.5, preserve_topology=True)

# Interactive map (uses Folium/Leaflet under the hood)
#gdf_simplified.explore()

## Step 2: Mask Each Raster

### Version using Rasterio

In [ ]:
# resampled_2017 = os.path.join(urban_dropbox,"ccap2017","resampled")

# raster_list = [f for f in os.listdir(raster_folder) if f.endswith('.TIF')] # list rasters in the directory

# if not raster_list:
#     print("No rasters found in the specified folder.")
# else:
#     for raster in raster_list:
#         raster_path = os.path.join(raster_folder, raster)
#         output_path = os.path.join(rasters_out, f"masked_{raster}")

#         # Check if the raster has already been processed
#         if os.path.exists(output_path):
#             print(f"Skipping {raster}, already processed.")
#             continue

#         # Process the raster
#         try:
#             with rasterio.open(raster_path) as src:
#                 # Apply the mask
#                 out_image, out_transform = mask(src, gdf_simplified, crop=True)
#                 out_meta = src.meta.copy()
                
#                 # Update metadata
#                 out_meta.update({
#                     "driver": "GTiff",
#                     "height": out_image.shape[1],
#                     "width": out_image.shape[2],
#                     "transform": out_transform
#                 })

#                 # Save the masked raster
#                 with rasterio.open(output_path, "w", **out_meta) as dest:
#                     dest.write(out_image)

#             print(f"Processed and saved {output_path}")
#         except Exception as e:
#             print(f"Failed to process {raster}. Error: {e}")

### Version using Xarray 
This one is better


In [ ]:
import rioxarray
import geopandas as gpd

#gdf = gpd.read_file(mask_vector_path)
#gdf = gdf.to_crs("EPSG:your_raster_crs")
resampled_2021 = os.path.join(urban_dropbox,"ccap2021","resampled")
# redefine rasters out
rasters_out = os.path.join(base_dir,"ccap2021","masked")

raster_list = [f for f in os.listdir(resampled_2021) if f.endswith('.tif')] # list rasters in the directory. NOTE for 2011, the rasters have a capitized "TIF" ending, but other years are "tif"

print(f"Found {len(raster_list)} rasters in {resampled_2021}")
print(raster_list)

for raster in raster_list:
    raster_path = os.path.join(resampled_2021, raster)
    output_path = os.path.join(rasters_out, f"masked_{raster}")

    if os.path.exists(output_path):
        print(f"Skipping {raster}, already processed.")
        continue

    try:
        # Load the raster as xarray DataArray
        da = rioxarray.open_rasterio(raster_path, masked=True, chunks={'x': 2048, 'y': 2048})

        # Ensure mask CRS matches raster
        if gdf_simplified.crs != da.rio.crs:
            gdf_mask = gdf_simplified.to_crs(da.rio.crs)
        else:
            gdf_mask = gdf_simplified

        # Debug prints: confirm spatial overlap
        print(f"\nProcessing: {raster}")
        print(f"Raster bounds: {da.rio.bounds()}")
        print(f"Mask bounds: {gdf_mask.total_bounds}")


        # Clip raster to simplified mask
        da_clipped = da.rio.clip(gdf_simplified.geometry, gdf_simplified.crs, drop=True, invert=False)

        # Check if anything remains after the clip
        if da_clipped.rio.width == 0 or da_clipped.rio.height == 0:
            print("⚠️ Clipped raster is empty — skipping save.")
            continue

        # Save clipped raster
        da_clipped.rio.to_raster(output_path)

    except Exception as e:
        print(f"Failed to process {raster}: {e}")


Found 21 rasters in C:\Users\mmtb\Donovan Lab Dropbox\Donovan Lab Team Folder\Donovan_Lab_GIS\Drivers\2_Urban_Runoff\large-raw-data\ccap2021\resampled
['utm4_hi_hawaii_2021_ccap_0.tif_3x3.tif', 'utm4_hi_hawaii_2021_ccap_1.tif_3x3.tif', 'utm4_hi_hawaii_2021_ccap_2.tif_3x3.tif', 'utm4_hi_hawaii_2021_ccap_3.tif_3x3.tif', 'utm4_hi_hawaii_2021_ccap_4.tif_3x3.tif', 'utm4_hi_hawaii_2021_ccap_5.tif_3x3.tif', 'utm4_hi_hawaii_2021_ccap_6.tif_3x3.tif', 'utm4_hi_hawaii_2021_ccap_7.tif_3x3.tif', 'utm4_hi_hawaii_2021_ccap_8.tif_3x3.tif', 'utm4_hi_honolulu_2021_ccap_0.tif_3x3.tif', 'utm4_hi_honolulu_2021_ccap_1.tif_3x3.tif', 'utm4_hi_honolulu_2021_ccap_2.tif_3x3.tif', 'utm4_hi_honolulu_2021_ccap_3.tif_3x3.tif', 'utm4_hi_kauai_2021_ccap_0.tif_3x3.tif', 'utm4_hi_kauai_2021_ccap_1.tif_3x3.tif', 'utm4_hi_kauai_2021_ccap_2.tif_3x3.tif', 'utm4_hi_kauai_2021_ccap_3.tif_3x3.tif', 'utm4_hi_maui_2021_ccap_0.tif_3x3.tif', 'utm4_hi_maui_2021_ccap_1.tif_3x3.tif', 'utm4_hi_maui_2021_ccap_2.tif_3x3.tif', 'utm4_hi_m

c:\Users\mmtb\miniconda3\envs\geoenv\Lib\site-packages\IPython\core\interactiveshell.py:3155: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  result = runner(coro)



Processing: utm4_hi_hawaii_2021_ccap_1.tif_3x3.tif
Raster bounds: (853108.187612232, 2094147.738903515, 898788.187612232, 2146051.738903515)
Mask bounds: [ 369737.83652392 2094715.79112281  940291.63879112 2458991.71049553]


c:\Users\mmtb\miniconda3\envs\geoenv\Lib\site-packages\IPython\core\interactiveshell.py:3155: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  result = runner(coro)



Processing: utm4_hi_hawaii_2021_ccap_2.tif_3x3.tif
Raster bounds: (897054.6380502565, 2095641.5458463833, 942765.6380502565, 2147581.5458463836)
Mask bounds: [ 369737.83652392 2094715.79112281  940291.63879112 2458991.71049553]


...

c:\Users\mmtb\miniconda3\envs\geoenv\Lib\site-packages\IPython\core\interactiveshell.py:3155: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  result = runner(coro)



Processing: utm4_hi_maui_2021_ccap_2.tif_3x3.tif
Raster bounds: (669125.9884090272, 2313007.3107041796, 743143.9884090273, 2357684.3107041796)
Mask bounds: [ 369737.83652392 2094715.79112281  940291.63879112 2458991.71049553]


c:\Users\mmtb\miniconda3\envs\geoenv\Lib\site-packages\IPython\core\interactiveshell.py:3155: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  result = runner(coro)



Processing: utm4_hi_maui_2021_ccap_3.tif_3x3.tif
Raster bounds: (743143.9930533085, 2313007.318212241, 817160.9930533085, 2357684.318212241)
Mask bounds: [ 369737.83652392 2094715.79112281  940291.63879112 2458991.71049553]


c:\Users\mmtb\miniconda3\envs\geoenv\Lib\site-packages\IPython\core\interactiveshell.py:3155: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  result = runner(coro)


#### Debugging failed rasters
Only run if needed

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Path to output folder with masked rasters - rasters out was named in the chunk above
output_folder = rasters_out  

# get files with 'south' in the name
south_rasters = [f for f in os.listdir(output_folder) if 'south' in f.lower() and f.endswith('.tif')] # replace with whatever needed to read in any outputs that seem weird

print(f"Found {len(south_rasters)} rasters containing 'south'")

# Loop through and analyze each raster, look at histograms to make sure values seem ok 
for raster_name in south_rasters:
    raster_path = os.path.join(output_folder, raster_name)
    print(f"\n🔍 Analyzing: {raster_name}")

    try:
        da = rioxarray.open_rasterio(raster_path, masked=True)
        data = da.squeeze().values  # removes 'band' if it's 1-band

        # Summary info
        print(f"Shape: {da.shape}, Dims: {da.dims}")
        print(f"Dtype: {da.dtype}")
        print(f"Nodata: {da.rio.nodata}")
        print(f"CRS: {da.rio.crs}")
        print(f"Bounds: {da.rio.bounds()}")

         # Clean up the data array for histogram
        data_flat = data[~np.isnan(data)].flatten()

        if data_flat.size == 0:
            print("⚠️ All values are nodata or masked — empty raster.")
        else:
            print(f"Min: {np.min(data_flat)}, Max: {np.max(data_flat)}")
            print(f"Sample unique values: {np.unique(data_flat)[:10]}")

            # Histogram
            plt.hist(data_flat, bins=50)
            plt.title(f"Histogram: {raster_name}")
            plt.xlabel("Pixel Value")
            plt.ylabel("Frequency")
            plt.grid(True)
            plt.show()

    except Exception as e:
        print(f"❌ Failed to load {raster_name}: {type(e).__name__} - {e}")


NameError: name 'rasters_out' is not defined

## Step 3: Calculate pixels within each watershed

Goal: To estimate the area of impervious surface in each watershed

Assumptions: Each pixel that is classfied as impervious is fully impervious and can be multiplied by the area of the pixel (resolution squared)

### Using Rasterio (OLD - XARRAY works better)

In [ ]:

# # Define file paths
# masked_rasters = os.path.abspath(r".\outputs")
# output_folder = os.path.abspath(r".\outputs")

# # Print out the paths to check if they are correct
# print(f"Raster folder path: {masked_rasters}")
# print(f"Output folder path: {output_folder}")

# # Initialize a DataFrame to store aggregated results
# aggregated_results = pd.DataFrame()


# # Process each raster in the folder
# for raster_file in os.listdir(masked_rasters):
#     if raster_file.endswith(".TIF"):  # Process only .tif files
#         print(f"Processing {raster_file}...")
#         raster_path = os.path.join(masked_rasters, raster_file)

#     try:  
#         # Open the raster file
#         with rasterio.open(raster_path) as src:
#             raster_data = src.read(1)  # Read the first band
#             raster_meta = src.meta  # Metadata for the raster

#         # Calculate zonal statistics
#         stats = zonal_stats(
#             sheds_gdf, # this doesn't have to be the geometries which is cool
#             raster_data,
#             affine=raster_meta['transform'],
#             stats=['count'],
#             nodata=raster_meta.get('nodata')  # Handle nodata values
#         )
#         print(f"Zonal stats for {raster_file}: {stats}")

#         # Add results to the aggregated DataFrame
#         temp_df = pd.DataFrame({
#             "zone_id": sheds_gdf["HUC12"].values,  # Add unique ID for zones
#             f"pixel_count_{raster_file}": [stat["count"] for stat in stats]
#         })
        
#         print(f"Temporary DataFrame for {raster_file}:\n{temp_df}")
        
#         aggregated_results = pd.merge(
#             aggregated_results, temp_df, on="zone_id", how="outer"
#         ) if not aggregated_results.empty else temp_df
        
#     except Exception as e:
#         print(f"Error processing {raster_file}: {e}")


# Save the aggregated results to a CSV file
#aggregated_results.to_csv(os.path.join(output_folder, "aggregated_pixel_counts.csv"), index=False)

#print("Processing complete. Results saved to the output folder.")


NameError: name 'os' is not defined

### Using Xarray and RioXarray

Rasterstats does not import correctly into the newest version of Python. You can probably get around this by just building it straight from Github. But I'm going to try a different method instead:

- Rasterize the sheds_gdf zones to match the raster’s grid.

- Use xarray to group values by zone ID.

- Count how many non-nodata pixels fall into each zone.



In [ ]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
import rioxarray
from rasterio.features import rasterize

# Inputs
masked_rasters = rasters_out
output_folder = os.path.abspath(r".\outputs")
zone_gdf = sheds_gdf  # this has to loaded and in memory

# Create empty data frame to store output
aggregated_results = pd.DataFrame()

# re code the HUC12 to simpler ids
zone_ids_str = zone_gdf["HUC12"].astype(str)
zone_id_map = {z: i+1 for i, z in enumerate(zone_ids_str)}  # +1 to avoid 0 as background
zone_gdf["zone_int"] = zone_ids_str.map(zone_id_map)

# view mapping
print("HUC12 to zone_int mapping:")
print(zone_gdf[["HUC12", "zone_int"]].drop_duplicates().sort_values("zone_int"))

# save as csv
mapping_df = zone_gdf[["HUC12", "zone_int"]].drop_duplicates()
mapping_df.to_csv("zone_id_mapping.csv", index=False)

for raster_file in os.listdir(masked_rasters):
    if not raster_file.lower().endswith("tif"):
        continue

    raster_path = os.path.join(masked_rasters, raster_file)
    print(f"\nProcessing: {raster_file}")

    try:
        # open raster
        da = rioxarray.open_rasterio(raster_path, masked = True).squeeze()
        nodata_val = da.rio.nodata
        
        # Open lazily and trim to zone bounding box
        # da = rioxarray.open_rasterio(raster_path, masked=True, chunks={"x": 2048, "y": 2048}).squeeze()
        # # Clip to bounding box of zones to reduce memory usage
        # bbox = zone_gdf.total_bounds
        # da = da.rio.clip_box(*bbox)
        
        # rasterize zone polygons to match raster shape
        transform = da.rio.transform()
        shape = da.shape

        geometry = zone_gdf.geometry.values
        zone_int_ids = zone_gdf["zone_int"].values
    

        zone_mask = rasterize(
            [(geom,value) for geom, value in zip(geometry, zone_int_ids)],
            out_shape=shape,
            transform = transform,
            fill  = "0",
            dtype = "int32"
        )

        # mask out only values that = 1
        raster_data = da.values
        target_mask = (raster_data ==1)

        # apply target mask to zone mask
        target_zone_ids = zone_mask[target_mask]

        # count pixels with valu e== z per zone
        unique, counts = np.unique(target_zone_ids, return_counts = True)
        zone_counts = dict(zip(unique, counts))

        # build the temporary data frame for this raster area
        temp_df = pd.DataFrame({
            "zone_id": zone_int_ids,
            f"pixel_count_{raster_file}": [zone_counts.get(zid,0) for zid in zone_int_ids]
        })
        print(temp_df.head())

        if aggregated_results.empty:
            aggregated_results = temp_df
        else:
            aggregated_results = pd.merge(aggregated_results, temp_df, on ="zone_id", how = "outer")


    except Exception as e:
        print(f"❌ Failed to process {raster_file}: {type(e).__name__} - {e}")





HUC12 to zone_int mapping:
            HUC12  zone_int
0    200700000103         1
1    200700000205         2
2    200700000102         3
3    200600000101         4
4    200600000303         5
..            ...       ...
168  200200000403       169
169  200100000903       170
170  200100000502       171
171  200100000302       172
172  200100000405       173

[173 rows x 2 columns]

Processing: masked_utm4_hi_hawaii_2021_ccap_0.tif_3x3.tif
   zone_id  pixel_count_masked_utm4_hi_hawaii_2021_ccap_0.tif_3x3.tif
0        1                                                  0        
1        2                                                  0        
2        3                                                  0        
3        4                                                  0        
4        5                                                  0        

Processing: masked_utm4_hi_hawaii_2021_ccap_1.tif_3x3.tif
   zone_id  pixel_count_masked_utm4_hi_hawaii_2021_ccap_1.tif_3x3.tif
0    

In [ ]:
# Save the aggregated results to a CSV file
aggregated_results.to_csv(os.path.join(base_dir,"outputs", "aggregated_pixel_counts_2021_resampled.csv"), index=False)


#print("Processing complete. Results saved to the output folder.")

## Validation 

I'm not sure all of the masking worked, so here I'll read in just one S. raster and count the pixels in one watershed and see if it matches what the CSV has. If not, then need to re-run with chunking. 

In [ ]:
import rioxarray
import geopandas as gpd
import numpy as np

# Choose a raster and zone to test
raster_path = os.path.join(masked_rasters, "masked_utm4_hi_honolulu_2021_ccap_3.tif")
test_zone_id = "200600000201"  # replace with a real HUC12
zone = zone_gdf[zone_gdf["HUC12"] == test_zone_id]

# Open raster and clip to zone
da = rioxarray.open_rasterio(raster_path, masked=True, chunks={"x": 1024, "y": 1024}).squeeze()
da_clipped = da.rio.clip(zone.geometry, zone.crs, drop=True)

# Count pixels equal to 1
pixel_array = da_clipped.values
count_1s = np.sum(pixel_array == 1)

print(f"Manual pixel count for zone {test_zone_id}: {count_1s}")


Manual pixel count for zone 200600000201: 1678269
